# Um die besten Hyperparameter zu finden, wird (mit Weights and Biases) das BERT-Modell immer wieder mit anderen Parametern feinjustiert.

In [ ]:
import wandb
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline, AutoModelForMaskedLM
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd
import os
pd.set_option('display.max_colwidth', None)
pd.options.display.max_rows = 100

from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive')

os.chdir("/content/drive/MyDrive/Daten")

df = pd.read_csv("kandis_cleaned.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df.rename(columns={"body":"text"}, inplace=True)

In [ ]:
df_codiert = pd.read_excel("Excel_eigencodierung_final.xlsx")
df_codiert.rename(columns={"nummer":"index"}, inplace=True)
df_codiert.shape[0]

In [ ]:
df = df.reset_index()
df = pd.merge(df_codiert, df, on=["text","index"], how="outer")
print(df.shape[0])
df = df.dropna(subset=["text","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]).copy(deep=True)
df.shape[0]

In [ ]:
print(df[["D_0"]].value_counts())
print(df[["D_1"]].value_counts())
print(df[["D_2"]].value_counts())
print(df[["D_3"]].value_counts())
print(df[["D_4"]].value_counts())
print(df[["D_5"]].value_counts())
print(df[["D_6"]].value_counts())
print(df[["D_7"]].value_counts())

In [ ]:
train_df, test_df = train_test_split(df[["text","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]], test_size=0.2, random_state=5460)
train_df, val_df = train_test_split(df[["text","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]], test_size=0.25, random_state=5460)

In [ ]:
test_df.to_csv("inhalt-TEST.csv")
train_df.to_csv("inhalt-TRAIN.csv")
val_df.to_csv("inhalt-VAL.csv")

In [ ]:
label_columns = ["D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]

train_labels = train_df[label_columns].values.tolist()
val_labels = val_df[label_columns].values.tolist()

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.25).astype(int)
    return {
        "precision": precision_score(labels, preds, average="macro"), #vllt. anderes average
        "recall": recall_score(labels, preds, average="macro"),
        "f1_score": f1_score(labels, preds, average="macro"),
    }

In [ ]:
###############################
# 2) Prepare Your Dataset
###############################
def prepare_data(
    train_csv_path,
    val_csv_path,
    tokenizer_model="deepset/gbert-large"
):
    """
    Loads, concatenates, and encodes training and validation data.
    Also computes class weights for imbalanced datasets.
    Returns train_dataset, val_dataset, label_encoder, and class_weights.
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_df = pd.read_csv(train_csv_path)[["text","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]]
    val_df = pd.read_csv(val_csv_path)[["text","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]]

    label_columns = ["D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]
    train_labels = train_df[label_columns].values.tolist()
    val_labels = val_df[label_columns].values.tolist()

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)

    # Tokenize text
    train_encodings = tokenizer(train_df['text'].tolist(), truncation=True, padding=True, max_length=512)
    val_encodings = tokenizer(val_df['text'].tolist(), truncation=True, padding=True, max_length=512)

    # Create Hugging Face Datasets
    train_dataset = Dataset.from_dict({
        "input_ids": train_encodings['input_ids'],
        "attention_mask": train_encodings['attention_mask'],
        "labels": train_labels
    })
    val_dataset = Dataset.from_dict({
        "input_ids": val_encodings['input_ids'],
        "attention_mask": val_encodings['attention_mask'],
        "labels": val_labels
    })

    return train_dataset, val_dataset, label_columns


In [ ]:
##############################
# 4) Define Training Function
##############################
def train_and_evaluate(config=None):
    """
    Main training function that:
    1) Initializes a wandb run.
    2) Prepares data and model.
    3) Trains the model with specified hyperparameters (from `config`).
    4) Evaluates and logs final results.
    """
    with wandb.init(config=config):
        # Access hyperparameters from wandb.config
        config = wandb.config

        # --------------------
        # A) Prepare Data
        # --------------------
        train_dataset, val_dataset, label_columns = prepare_data(
            train_csv_path="inhalt-TRAIN.csv",
            val_csv_path="inhalt-VAL.csv",
            tokenizer_model="deepset/gbert-large"
        )

        # --------------------
        # B) Model & Trainer
        # --------------------
        model = AutoModelForSequenceClassification.from_pretrained(
            "deepset/gbert-large",
            num_labels=8,  # binary classification
            problem_type="multi_label_classification"
        )

        training_args = TrainingArguments(
            output_dir='./results',                     # Directory to save training outputs like checkpoints and logs
            num_train_epochs=config.num_train_epochs,  # Number of epochs to train the model, set from the configuration
            per_device_train_batch_size=config.batch_size,  # Batch size for training on each device (e.g., GPU or CPU)
            per_device_eval_batch_size=config.batch_size,   # Batch size for evaluation on each device
            warmup_steps=config.warmup_steps,          # Number of steps for learning rate warm-up to stabilize initial training
            weight_decay=config.weight_decay,          # Weight decay (L2 regularization) for optimizer to prevent overfitting
            learning_rate=config.learning_rate,        # Initial learning rate for the optimizer, set from the configuration
            eval_strategy="epoch",                     # Evaluate the model at the end of each epoch
            save_strategy="epoch",                     # Save model checkpoints at the end of each epoch
            metric_for_best_model="f1_score",          # Specify the metric to use for selecting the best model (F1 score here)
            greater_is_better=True,                    # Indicate that higher values of the F1 score are better
            load_best_model_at_end=True,               # Automatically load the model with the best F1 score after training
            logging_strategy="steps",                  # Log training progress at regular step intervals
            logging_steps=10,                          # Log metrics and progress every 10 training steps
            report_to='wandb'                          # Report logs and metrics to Weights & Biases (wandb) for tracking
        )


        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics
)

        # --------------------
        # C) Train & Evaluate
        # --------------------
        trainer.train()
        final_results = trainer.evaluate()

        # --------------------
        # D) Log Results
        # --------------------
        wandb.log({"final_results": final_results})
        print("Final results:", final_results)

        wandb.finish()

In [ ]:
#####################################
# 5) Example: Running a Hyperparam Sweep
#####################################
"""
Example of defining a hyperparameter sweep in wandb.
This sweep will try different learning rates, weight decays, etc.
You can adjust the ranges to your liking.
"""
sweep_config = {
    "method": "bayes",  # Use Bayesian optimization to find the best combination of parameters
    "metric": {
        "name": "eval_f1_score",  # Evaluation metric to be optimized (F1 score in this case)
        "goal": "maximize"        # Aim to maximize the F1 score during the sweep
    },
    "parameters": {
        "num_train_epochs": {
            "values": [4, 6, 8]
        },
        "learning_rate": {
            "values": [2e-5, 4e-5, 6e-5]
        },
        "weight_decay": {
            "values": [0.01, 0.03, 0.05]
        },
        "warmup_steps": {
            "values": [0, 100]
        },
        "batch_size": {
            "values": [8]
        }
    }
}

# Create sweep
sweep_id = wandb.sweep(sweep_config, project="gbert_class_multi")
# Start the sweep
wandb.agent(sweep_id, function=train_and_evaluate)